In [ ]:
from pathlib import Path
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

from mihms.prep import prms as pprms
from mihms.models.models import PrmsModel
from mihms.output import PrmsOutput

In [ ]:
calperiod = slice('2020-01-01', '2024-12-31')
calperiod_size = len(pd.date_range('2020-01-01', '2024-12-31', freq='D'))

def run_prms(in_params: tuple | None = None) -> np.ndarray:
    """Changes parameters and runs a PRMS Model.

    If user provides a dictionary of parameters whose item value is a list of numpy arrays where
    index[0] = new parameter values and index[1] = paired PRMS subbasin ID's, this function will write
    the new parameter values and run the model. If no parameters are provided, the model will run with the
    parameters that exist in the PRMS parameter file as-is.

    Args:
        in_params:
            Must be a tuple of (integer, parameter dictionary).
    
    Returns:
        The model run results for PRMS Subbasins.
    """

    m_wd = Path('D:/Modeling/GSFLOW/PRMS_Projects/Upper Yellowstone/UYRHMv100')

    if in_params is None:
        prmscont = pprms.CreateControlFile.load_from_prms_controlfile(m_wd / 'UYstudy.control')
        mod = PrmsModel(prmscont)
        mod.run_model(silent=True, report=False)
    else:
        shutil.copy((m_wd / 'UYparams.param'), (m_wd / f'UYparams_{in_params[0]}.param'))
        prmscont = pprms.CreateControlFile(prms_exe=Path('C:/Users/CNB968/OneDrive - MT/Modeling/GSFLOW/PRMS/prms_6.0.0/bin/prms.exe'),
                                    prms_version='6.0.0',
                                    start_time='2018-01-01',
                                    end_time='2024-12-31',
                                    data_files=m_wd / 'UYdata.data',
                                    param_files=m_wd / f'UYparams_{in_params[0]}.param',
                                    output_path=m_wd,
                                    file_name=f'UYstudy_{in_params[0]}.control')
        prmscont.select_output_variables(['sub_cfs', 'sub_inq'])
        prmscont.control_obj.set_values('strmflow_module', ['muskingum_lake'])
        prmscont.control_obj.set_values('cascadegw_flag', [2])
        prmscont.save_controlfile()
    
        mod = PrmsModel(prmscont)

        for k,v in in_params[1].items():
            mod.parameters.change_parameter_by_subbasin(k, v[0], v[1])
        mod.parameters.write_paramfile()
        mod.run_model(silent=True, report=False)
    res = PrmsOutput(mod.control)

    subout = res.get_subbasin_output()
    sim_arr = np.empty((calperiod_size, v[1].size))
    for i, s in enumerate(v[1]):
        simvals = subout.sub_inq.sel(subbasin=s, time=calperiod).values
        sim_arr[:,i] = simvals

    (m_wd / f'UYparams_{in_params[0]}.param').unlink()
    (m_wd / f'UYstudy_{in_params[0]}.control').unlink()
    shutil.rmtree(m_wd / prmscont._out_pth)

    return sim_arr


def parallel_threads(in_iter: list | np.ndarray, workers: int = os.cpu_count()):
    # Initialize the progress bar
    with tqdm(total=total_tasks, desc="Processing") as pbar:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks
            futures = {executor.submit(dummy_task, i): i for i in range(total_tasks)}
            
            # As each task is submitted, it occupies a worker slot
            # We estimate running count as (submitted/active minus completed) capped at max_workers
            for future in as_completed(futures):
                # A task just finished, update counts
                tasks_completed += 1
                active_running = min(max_workers, len(futures) - tasks_completed)
                
                # Update progress bar and postfix description with running count
                pbar.update(1)
                pbar.set_postfix(running=active_running)
                
                # Retrieve result if needed
                _ = future.result()

In [4]:
run_prms()

------------------------------------
Reading parameter file : UYparams.param
------------------------------------


AttributeError: 'PrmsModel' object has no attribute 'model'

In [7]:
run_prms((0, {'gwflow_coef': [np.array([0.31]), np.array([1])]}))

------------------------------------
Reading parameter file : UYparams_0.param
------------------------------------


array([[6.0635],
       [5.5686],
       [5.1114],
       ...,
       [8.797 ],
       [8.3213],
       [7.9042]], shape=(1827, 1))

In [11]:
print(np.array([[1,2,3,4], [1,2,3,4]]))

[[1 2 3 4]
 [1 2 3 4]]


In [15]:
import dask.array as da
Ls = []
for i in np.arange(1000):
    Ls.append(da.ones((1800, 45000), chunks=(1, 5000)))

In [17]:
da.stack(Ls, axis=0)

dask.array<stack, shape=(1000, 1800, 45000), dtype=float64, chunksize=(1, 1, 5000), chunktype=numpy.ndarray>